In [1]:
import json
import os
import sys
from dataclasses import dataclass, field
from pathlib import Path
from typing import Literal

import optuna
import wandb
from dotenv import load_dotenv

sys.path.append(os.path.abspath("../.."))

import src.utils.run_optuna as op
from src.utils.optuna_objective import create_objective

### Config

In [2]:
load_dotenv(dotenv_path="../../.env")


@dataclass
class Config:
    # Data / CV
    model_name: str = "realmlp"
    data_id: str = "057"
    n_folds: int = 5
    seed: int = 42
    fold_idx: int = 0

    # Optuna
    n_trials: int = 1
    direction: str = "maximize"
    sampler: str = "tpe"  # tpe / random
    pruner: str = "median"  # median / none

    # Initial params
    use_initial: Literal["never", "manual"] = "never"
    initial_param_sources: list[tuple[str, int]] = field(default_factory=list)   # (study_name, n_trial) 例: ("xgb-001", 1)

    # Storage
    storage: str = "sqlite:////home/hanse/kaggle/binary-bank/artifacts/optuna/optuna.db"

    # Option
    opts: dict = field(default_factory=dict)


cfg = Config()
cfg.initial_param_sources = [("lgbm-057", 16)]

# W&B
wandb_project = os.environ.get("COMPETITION_NAME")
wandb.login(key=os.environ.get("WANDB_API_KEY"))

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /home/hanse/.netrc
wandb: Currently logged in as: kaitookano (kaitookano-waseda-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

### Build & Run

In [3]:
# --- helper: initial params loader ---
def load_initial_params(sources: list[tuple[str, int]]) -> list[dict]:
    loaded = []
    for study, n_trial in sources:
        path = Path(f"../../artifacts/optuna/{study}/trl{n_trial}.json")
        with path.open("r") as f:
            params = json.load(f)["params"]
        loaded.append(params)
    return loaded


# sampler / pruner factory
def build_sampler(name, seed):
    if name == "tpe":
        return optuna.samplers.TPESampler(n_startup_trials=15, seed=seed)
    elif name == "random":
        return optuna.samplers.RandomSampler(seed=seed)
    else:
        raise ValueError(f"unknown sampler: {name}")


def build_pruner(name):
    if name == "median":
        return optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=1000)
    elif name == "none":
        return optuna.pruners.NopPruner()
    else:
        raise ValueError(f"unknown pruner: {name}")


objective = create_objective(
    cfg.model_name,
    cfg.data_id,
    seed=cfg.seed,
    n_folds=cfg.n_folds,
    fold_idx=cfg.fold_idx,
    wandb_project=wandb_project,
    study_name=f"{cfg.model_name}-{cfg.data_id}",
    opts=cfg.opts
)

sampler = build_sampler(cfg.sampler, cfg.seed)
pruner = build_pruner(cfg.pruner)

initial_params = None
if cfg.use_initial == "manual":
    initial_params = load_initial_params(cfg.initial_param_sources)

op.run_optuna_search(
    objective,
    n_trials=cfg.n_trials,
    direction=cfg.direction,
    study_name=f"{cfg.model_name}-{cfg.data_id}",
    storage=cfg.storage,
    sampler=sampler,
    pruner=pruner,
    initial_params=initial_params
)

[I 2025-10-07 23:28:06,130] Using an existing study with name 'realmlp-057' instead of creating a new one.


[initial] none


  0%|          | 0/1 [00:00<?, ?it/s]

Fold Col: 5fold-s42
Free CPU Mem: 15.88 GB
Free GPU Mem: 6.81 GB


/home/hanse/kaggle/binary-bank/src/models/realmlp_cv_trainer.py:378: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  model.fit(X_train, y_train)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA GeForce RTX 4060 Ti') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
`Trainer.fit` stopped: `max_epochs=256` reached.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Logloss Valid: 1.90791
AUC Valid: 0.85903

Total Runtime: 00:33:36
Free CPU Mem: 13.27 GB
Free GPU Mem: 5.0 GB


auc_f1,0.85903
logloss_f1,1.90791
runtime_f1,33.61486


[I 2025-10-08 00:03:41,406] Trial 7 finished with value: 1.9079107193972682 and parameters: {}. Best is trial 6 with value: 1.9526048495997737.
✅ Message sent.


In [1]:
from pytabkit import RealMLP_TD_Classifier
help(RealMLP_TD_Classifier)

Help on class RealMLP_TD_Classifier in module pytabkit.models.sklearn.sklearn_interfaces:

class RealMLP_TD_Classifier(RealMLPConstructorMixin, pytabkit.models.sklearn.sklearn_base.AlgInterfaceClassifier)
 |  RealMLP_TD_Classifier(
 |      device: Optional[str] = None,
 |      random_state: Union[int, numpy.random.mtrand.RandomState, NoneType] = None,
 |      n_cv: int = 1,
 |      n_refit: int = 0,
 |      n_repeats: int = 1,
 |      val_fraction: float = 0.2,
 |      n_threads: Optional[int] = None,
 |      tmp_folder: Union[str, pathlib._local.Path, NoneType] = None,
 |      verbosity: int = 0,
 |      train_metric_name: Optional[str] = None,
 |      val_metric_name: Optional[str] = None,
 |      n_epochs: Optional[int] = None,
 |      batch_size: Optional[int] = None,
 |      predict_batch_size: Optional[int] = None,
 |      hidden_sizes: Union[List[int], Literal['rectangular'], NoneType] = None,
 |      n_hidden_layers: Optional[int] = None,
 |      hidden_width: Optional[int] = N